In [ ]:
import os

# Create folder structure
folders = [
    "project/scripts",
    "project/data",
    "project/results"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Create Snakefile
snakefile = """\
configfile: "config.yaml"

rule all:
    input:
        "results/metrics.json"

rule preprocess:
    input:
        "data/raw_data.csv"
    output:
        "results/cleaned_data.csv"
    script:
        "scripts/preprocess.py"

rule train:
    input:
        "results/cleaned_data.csv"
    output:
        "results/model.pkl"
    params:
        model_type = config["model_type"],
        target = config["target_column"]
    script:
        "scripts/train_model.py"

rule evaluate:
    input:
        model = "results/model.pkl",
        data = "results/cleaned_data.csv"
    output:
        "results/metrics.json"
    script:
        "scripts/evaluate_model.py"
"""

# config.yaml
config_yaml = """\
target_column: target
model_type: random_forest
"""

# preprocess.py
preprocess_py = """\
import pandas as pd

df = pd.read_csv(snakemake.input[0])
df = df.dropna()
df.to_csv(snakemake.output[0], index=False)
"""

# train_model.py
train_model_py = """\
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

df = pd.read_csv(snakemake.input[0])
target = snakemake.params["target"]
X = df.drop(columns=[target])
y = df[target]

if snakemake.params["model_type"] == "random_forest":
    model = RandomForestClassifier()
else:
    model = LogisticRegression()

model.fit(X, y)
with open(snakemake.output[0], "wb") as f:
    pickle.dump(model, f)
"""

# evaluate_model.py
evaluate_model_py = """\
import pandas as pd
import pickle
import json
from sklearn.metrics import roc_auc_score

df = pd.read_csv(snakemake.input["data"])
with open(snakemake.input["model"], "rb") as f:
    model = pickle.load(f)

y = df["target"]
X = df.drop(columns=["target"])
y_pred = model.predict_proba(X)[:, 1]

auc = roc_auc_score(y, y_pred)
with open(snakemake.output[0], "w") as f:
    json.dump({"auc": auc}, f)
"""

# Save files
with open("project/Snakefile", "w") as f:
    f.write(snakefile)

with open("project/config.yaml", "w") as f:
    f.write(config_yaml)

with open("project/scripts/preprocess.py", "w") as f:
    f.write(preprocess_py)

with open("project/scripts/train_model.py", "w") as f:
    f.write(train_model_py)

with open("project/scripts/evaluate_model.py", "w") as f:
    f.write(evaluate_model_py)

" Project template created! You're ready to run Snakemake."


### 📁 Folder Structure

```
project/
├── Snakefile                  ← defines your workflow
├── config.yaml                ← parameters (model type, target col)
├── data/
│   └── raw_data.csv           ← input your data here
├── results/
│   ├── cleaned_data.csv       ← cleaned data output
│   ├── model.pkl              ← trained model
│   └── metrics.json           ← evaluation (e.g., AUC)
└── scripts/
    ├── preprocess.py
    ├── train_model.py
    └── evaluate_model.py
```

---

### 🧪 Next Steps

1. **Put your input CSV** (e.g., from cBioPortal or any source) into:

```
project/data/raw_data.csv
```

Make sure it has a column named `target` (or change it in `config.yaml`).

2. **Go to the `project/` folder** and run the workflow:

```bash
cd project
snakemake --cores 1
```

You’ll get:

* Cleaned data
* A trained model (default: Random Forest)
* AUC score saved in `results/metrics.json`

---

### Extras

#### ✅ Use `conda` for reproducibility

Add this to the top of each rule:

```python
conda: "envs/ml.yaml"
```

And create `envs/ml.yaml` like:

```yaml
name: ml
channels:
  - conda-forge
  - defaults
  - bioconda
dependencies:
  - python=3.10
  - pandas
  - scikit-learn
```

Then run:

```bash
snakemake --use-conda --cores 1
```

---

### Add parallel models, seeds, etc.

Expand this to run:

* Multiple model types (RF, SVM, etc.)
* Cross-validation loops
* Seeds or bootstraps

---
Next:
* Add conda/env support
* Add multiple models or hyperparameter tuning
* Help convert a specific Python notebook into this format



✅ Your **Snakemake ML project template** is ready! Here's what you now have:

---

### 📁 Folder Structure

```
project/
├── Snakefile                  ← defines your workflow
├── config.yaml                ← parameters (model type, target col)
├── data/
│   └── raw_data.csv           ← input your data here
├── results/
│   ├── cleaned_data.csv       ← cleaned data output
│   ├── model.pkl              ← trained model
│   └── metrics.json           ← evaluation (e.g., AUC)
└── scripts/
    ├── preprocess.py
    ├── train_model.py
    └── evaluate_model.py
```

---

### 🧪 Next Steps

1. **Put your input CSV** (e.g., from cBioPortal or any source) into:

```
project/data/raw_data.csv
```

Make sure it has a column named `target` (or change it in `config.yaml`).

2. **Go to the `project/` folder** and run the workflow:

```bash
cd project
snakemake --cores 1
```

You’ll get:

* Cleaned data
* A trained model (default: Random Forest)
* AUC score saved in `results/metrics.json`

---

### 🔧 Extras You Can Add

#### ✅ Use `conda` for reproducibility

Add this to the top of each rule:

```python
conda: "envs/ml.yaml"
```

And create `envs/ml.yaml` like:

```yaml
name: ml
channels:
  - conda-forge
  - defaults
  - bioconda
dependencies:
  - python=3.10
  - pandas
  - scikit-learn
```

Then run:

```bash
snakemake --use-conda --cores 1
```

---

### 🔁 Add parallel models, seeds, etc.

We can easily expand this to run:

* Multiple model types (RF, SVM, etc.)
* Cross-validation loops
* Seeds or bootstraps

---

Would you like me to:

* Add conda/env support?
* Add multiple models or hyperparameter tuning?
* Help convert a specific Python notebook into this format?